In [ ]:
from collections import Counter, defaultdict
import math

class NPHMM:
    def __init__(self, k=1.0):
        self.k = k
        self.tags = []
        self.vocab = set()
        self.start_counts = Counter()
        self.transition_counts = defaultdict(Counter)
        self.emission_counts = defaultdict(Counter)
        self.tag_counts = Counter()

    def fit(self, train_data):
        for sentence in train_data:
            if not sentence:
                continue

            previous_tag = None

            for i, (word, tag) in enumerate(sentence):
                self.vocab.add(word)
                self.tag_counts[tag] += 1
                self.emission_counts[tag][word] += 1

                if i == 0:
                    self.start_counts[tag] += 1

                if previous_tag is not None:
                    self.transition_counts[previous_tag][tag] += 1

                previous_tag = tag

        self.tags = sorted(self.tag_counts.keys())
        self.vocab.add("<UNK>")
        self.num_tags = len(self.tags)
        self.vocab_size = len(self.vocab)
        self.num_sentences = sum(bool(s) for s in train_data)
        return self

    def emission_logprob(self, word, tag):
        # Add-K smoothing for word emissions
        count = self.emission_counts[tag][word]
        denominator = self.tag_counts[tag] + self.k * self.vocab_size
        return math.log(
            (count + self.k) / denominator
        )

    def transition_logprob(self, previous_tag, tag):
        # Add-K smoothing for tag transitions
        count = self.transition_counts[previous_tag][tag]
        denominator = self.tag_counts[previous_tag] + self.k * self.num_tags
        return math.log(
            (count + self.k) / denominator
        )

    def start_logprob(self, tag):
        count = self.start_counts[tag]
        denominator = self.num_sentences + self.k * self.num_tags
        return math.log(
            (count + self.k) / denominator
        )

    # Task 1: Forward algorithm
    def forward_logprob(self, words):
        if not words:
            return 0.0

        alpha = {}

        for tag in self.tags:
            alpha[tag] = (
                self.start_logprob(tag)
                + self.emission_logprob(words[0], tag)
            )

        for word in words[1:]:
            next_alpha = {}

            for tag in self.tags:
                scores = [
                    alpha[previous_tag]
                    + self.transition_logprob(previous_tag, tag)
                    for previous_tag in self.tags
                ]

                # log-sum-exp avoids numerical underflow
                maximum = max(scores)
                total = maximum + math.log(
                    sum(math.exp(score - maximum) for score in scores)
                )

                next_alpha[tag] = (
                    total + self.emission_logprob(word, tag)
                )

            alpha = next_alpha

        scores = list(alpha.values())
        maximum = max(scores)
        return maximum + math.log(
            sum(math.exp(score - maximum) for score in scores)
        )

    # Task 2: Viterbi decoding
    def viterbi(self, words):
        if not words:
            return []

        dp = [{}]
        backpointer = [{}]

        for tag in self.tags:
            dp[0][tag] = (
                self.start_logprob(tag)
                + self.emission_logprob(words[0], tag)
            )
            backpointer[0][tag] = None

        for i in range(1, len(words)):
            dp.append({})
            backpointer.append({})

            for tag in self.tags:
                candidates = [
                    (
                        dp[i - 1][previous_tag]
                        + self.transition_logprob(previous_tag, tag)
                        + self.emission_logprob(words[i], tag),
                        previous_tag
                    )
                    for previous_tag in self.tags
                ]

                best_score, best_previous_tag = max(candidates)
                dp[i][tag] = best_score
                backpointer[i][tag] = best_previous_tag

        last_tag = max(dp[-1], key=dp[-1].get)
        predicted_tags = [last_tag]

        for i in range(len(words) - 1, 0, -1):
            last_tag = backpointer[i][last_tag]
            predicted_tags.append(last_tag)

        predicted_tags.reverse()
        return predicted_tags

In [ ]:

def load_tagged_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            # A blank line marks the end of a sentence
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue

            parts = line.split()

            # Each nonblank line should contain a word and its tag
            if len(parts) >= 2:
                word = parts[0]
                tag = parts[-1]
                current_sentence.append((word, tag))

    # Include the last sentence if there is no trailing blank line
    if current_sentence:
        sentences.append(current_sentence)

    return sentences


train_data = load_tagged_data("/content/train_NP.txt")
test_data = load_tagged_data("/content/test_NP.txt")


print("Number of training sentences:", len(train_data))
print("First sentence:", train_data[0])

print("Number of testing sentences:", len(test_data))
print("First sentence:", test_data[0])


Number of training sentences: 823
First sentence: [('Rockwell', 'B-NP'), ('International', 'I-NP'), ('Corp.', 'I-NP'), ("'s", 'B-NP'), ('Tulsa', 'I-NP'), ('unit', 'I-NP'), ('said', 'O'), ('it', 'B-NP'), ('signed', 'O'), ('a', 'B-NP'), ('tentative', 'I-NP'), ('agreement', 'I-NP'), ('extending', 'O'), ('its', 'B-NP'), ('contract', 'I-NP'), ('with', 'O'), ('Boeing', 'B-NP'), ('Co.', 'I-NP'), ('to', 'O'), ('provide', 'O'), ('structural', 'B-NP'), ('parts', 'I-NP'), ('for', 'O'), ('Boeing', 'B-NP'), ("'s", 'B-NP'), ('747', 'I-NP'), ('jetliners', 'I-NP'), ('.', 'O')]
Number of testing sentences: 77
First sentence: [('Confidence', 'B-NP'), ('in', 'O'), ('the', 'B-NP'), ('pound', 'I-NP'), ('is', 'O'), ('widely', 'O'), ('expected', 'O'), ('to', 'O'), ('take', 'O'), ('another', 'B-NP'), ('sharp', 'I-NP'), ('dive', 'I-NP'), ('if', 'O'), ('trade', 'B-NP'), ('figures', 'I-NP'), ('for', 'O'), ('September', 'B-NP'), (',', 'O'), ('due', 'O'), ('for', 'O'), ('release', 'B-NP'), ('tomorrow', 'B-NP'), ('

In [ ]:

hmm = NPHMM(k=1.0)
hmm.fit(train_data)

# Task 1: Probability of each test sentence
for sentence in test_data:
    words = [word for word, tag in sentence]
    log_probability = hmm.forward_logprob(words)

    print("Sentence:", " ".join(words))
    print("Log probability:", log_probability)

# Task 2: Predict NP tags
for sentence in test_data:
    words = [word for word, tag in sentence]
    predicted_tags = hmm.viterbi(words)

    print("Sentence:", " ".join(words))
    print("Predicted tags:", predicted_tags)

Sentence: Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
Log probability: -263.67713340609185
Sentence: Chancellor of the Exchequer Nigel Lawson 's restated commitment to a firm monetary policy has helped to prevent a freefall in sterling over the past week .
Log probability: -188.57167926654162
Sentence: But analysts reckon underlying support for sterling has been eroded by the chancellor 's failure to announce any new policy measures in his Mansion House speech last Thursday .
Log probability: -213.48865720067374
Sentence: This has increased the risk of the government being forced to increase base rates to 16 % from their current 15 % level to defend the pound , economists and foreign exchange market analysts say .
Log probability: -243.8873568485172
Sentence: `` The risks for sterling of a bad trade figure are very h

In [ ]:
!pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 38.0 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


In [ ]:
!pip install seqeval

  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=a9f95155fafa335fcdcdca3f4c170f224123207e5ec3fb374be3de876e4050d8
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
from seqeval.metrics import classification_report, f1_score

y_true = []
y_pred = []

for sentence in test_data:
    words = [word for word, tag in sentence]
    true_tags = [tag for word, tag in sentence]

    y_true.append(true_tags)
    y_pred.append(hmm.viterbi(words))

print(classification_report(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

              precision    recall  f1-score   support

          NP       0.64      0.73      0.68       475

   micro avg       0.64      0.73      0.68       475
   macro avg       0.64      0.73      0.68       475
weighted avg       0.64      0.73      0.68       475

F1-score: 0.6818181818181819
